# Aula 1 · Ensinar a máquina com exemplos

**10 minutos (11:40–11:50) · aula guiada, em pares** · a seguir: a Prática 1

**O que vamos fazer:** ensinar o computador a reconhecer a água numa imagem de satélite. Não vamos escrever a regra: damos-lhe **exemplos com a resposta certa** e ele descobre a regra sozinho. É isto o Machine Learning.

**Os dados:** `incomati_2026_cheia.npz` (a imagem do satélite da cheia de 24/01/2026) e `exemplos_da_equipa.json` (15 pontos de água e 15 de não água, com a resposta certa).  
**Treino e teste:** ainda não separamos: a árvore aprende com todos os exemplos e medimos o acerto nos mesmos. A Aula 2 mostra porque é que isso engana.  
**O algoritmo:** árvore de decisão (classificação: água ou não água).

| Palavra nova | O que quer dizer |
| --- | --- |
| **exemplo** | um caso que mostramos ao computador (aqui, um píxel da imagem) |
| **característica** | o que se mede em cada exemplo (aqui, as 5 cores medidas pelo satélite) |
| **rótulo** | a resposta certa de cada exemplo (aqui, 1 = água e 0 = não água) |
| **modelo** | a regra que o computador aprende com os exemplos |
| **treinar** | dar os exemplos ao modelo para ele aprender |
| **prever** | usar o modelo para dar a resposta num caso novo |

**Como trabalhar:** esta é a **aula guiada**: o facilitador mostra no projector e vocês acompanham.
Cliquem numa célula de código (as cinzentas) e carreguem em **Shift + Enter** para a correr, sempre **por ordem**, de
cima para baixo. A seguir vem a **prática**, em que são vocês a escrever o código. Guardem com **Ctrl + S**.

## Passo 1 · Abrir as ferramentas

As ferramentas abrem as imagens, desenham os mapas e corrigem os exercícios.

In [ ]:
from ferramentas import *      # as ferramentas da aula: corram sem mudar nada
print("Ferramentas prontas.")

## Passo 2 · A imagem do satélite

O satélite **Sentinel-2** fotografou o rio Incomáti, em Marracuene, a **24 de Janeiro de 2026**, no meio da cheia. Cada quadradinho da imagem é um **píxel** de 30 m × 30 m. Os círculos são pontos de **água** marcados pela equipa; os quadrados são pontos de **não água** (terra, plantas, casas).

In [ ]:
cheia = carregar("incomati_2026_cheia.npz")
pontos = pontos_da_equipa(2026)
ver_imagem(cheia, "Marracuene, 24 de Janeiro de 2026", pontos["agua"], pontos["nao_agua"])

## Passo 3 · Uma imagem é uma tabela de números

O computador não vê cores: vê números. Em cada píxel, o satélite mede 5 bandas: azul, verde, vermelho e dois infravermelhos que os nossos olhos não vêem, o **NIR** e o **SWIR**. Comparemos um píxel de água com um de terra.

In [ ]:
l, c = pontos["agua"][0]            # um ponto de água: linha e coluna
l2, c2 = pontos["nao_agua"][0]      # um ponto de terra
pd.DataFrame({"píxel de água": [cheia[b][l, c] for b in BANDAS],
              "píxel de terra": [cheia[b][l2, c2] for b in BANDAS]}, index=BANDAS)

Olhem para a última linha, o **SWIR**: na água é quase 0, porque a água absorve este infravermelho; na terra é muito maior. Guardem esta ideia.

## Passo 4 · Os exemplos, com os rótulos

À volta de cada ponto marcado tiramos 25 píxeis (um quadrado de 5 × 5). Fica uma tabela: uma linha por exemplo, as 5 **características** e o **rótulo** (1 = água, 0 = não água).

In [ ]:
exemplos = tabela_de_exemplos(cheia, pontos["agua"], pontos["nao_agua"])
print("Número de exemplos:", len(exemplos))
print("De água:", (exemplos["rótulo"] == 1).sum(), "   De não água:", (exemplos["rótulo"] == 0).sum())
exemplos.sample(8, random_state=1)        # 8 linhas da tabela, ao acaso

## Passo 5 · Uma regra feita por nós

Antes da máquina, tentemos nós. A nossa regra: **se o SWIR for menor do que 0,10, é água**. Quantos exemplos acerta? (Em Python, as décimas escrevem-se com ponto: 0.10.)

In [ ]:
limite = 0.10
resposta_da_regra = (exemplos["swir"] < limite).astype(int)      # 1 = água, 0 = não água
print("A regra acerta", percentagem(acerto(exemplos["rótulo"], resposta_da_regra)), "dos exemplos.")

## Passo 6 · A máquina aprende a regra sozinha

Agora damos os exemplos a uma **árvore de decisão**. Ela procura a pergunta que melhor separa a água da não água. Isto é **treinar** o modelo. `X` são as características e `y` são os rótulos.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

X = exemplos[BANDAS]            # as características
y = exemplos["rótulo"]          # as respostas certas
arvore = DecisionTreeClassifier(max_depth=2, random_state=0)
arvore.fit(X, y)                # treinar: aprender com os exemplos
print(export_text(arvore, feature_names=BANDAS).replace("class: 1", "ÁGUA").replace("class: 0", "NÃO ÁGUA"))
print("A árvore acerta", percentagem(acerto(y, arvore.predict(X))), "dos exemplos.")

A árvore escolheu sozinha o **SWIR**, como nós, mas encontrou um limite melhor. Ninguém lhe disse que o SWIR era importante: aprendeu com os exemplos.

## Passo 7 · Prever: o mapa da água

O modelo treinado pode responder para **todos** os píxeis da imagem, também os que nunca viu. Isto é **prever**.

In [ ]:
mapa_cheia = prever_mapa(arvore, cheia)
ver_mapa(mapa_cheia, cheia, "Água segundo o modelo, 24 de Janeiro de 2026")
print("Água no distrito:", numero(hectares(mapa_cheia, cheia)), "hectares")

## O que aprendemos

- O computador aprende com **exemplos** que têm a resposta certa (os **rótulos**).
- Em cada exemplo, olha para as **características**: aqui, as 5 bandas do satélite.
- Ao **treinar**, o **modelo** encontra a regra sozinho: aqui, «SWIR baixo é água».
- Depois **prevê** a resposta em casos que nunca viu: todos os píxeis da imagem.
- Atenção: medimos o acerto nos **mesmos** exemplos do treino. Na Aula 2 vemos porque é que isso não chega.

## Se aparecer um erro

Leiam a **última linha** da mensagem vermelha.

| A última linha diz | O que fazer |
| --- | --- |
| `NameError: name '...' is not defined` | Saltaram uma célula. Corram as células por ordem, a partir do Passo 1. |
| `FileNotFoundError: Não encontrei ...` | Abram o notebook a partir da pasta `Workshop_2/2_pratica`. |
| `ModuleNotFoundError` | Falta uma biblioteca neste computador: chamem um facilitador. |
| `NotFittedError` ou «O modelo ainda não foi treinado» | Falta treinar o modelo: corram a célula com `.fit(...)`. |
| `SyntaxError` | Falta uma aspa, um parêntese ou uma vírgula. Comparem com o exemplo. |
| «Falta preencher» | Troquem os `...` pela vossa resposta. |

Se nada resultar: menu **Kernel**, depois **Restart Kernel and Run All Cells**.